### Notebook 02 — Neural Bigram Language Model

#### Introdução

No notebook anterior, construímos um modelo de linguagem baseado em Bigramas utilizando tabelas de contagens simples.

Neste notebook daremos o próximo passo: substituiremos essa tabela estática por uma **rede neural**.

A diferença fundamental é como chegamos às probabilidades:

- **Notebook 01**: contamos quantas vezes cada par de caracteres aparece e dividimos
- **Notebook 02**: começamos com pesos aleatórios e os *ajustamos iterativamente* a partir dos dados

Para isso usaremos a biblioteca **PyTorch**, que facilita a construção e o treinamento de redes neurais em Python.

---

#### O que vamos aprender

Ao final deste notebook você deverá entender:

- O que é um **tensor** e como organizar dados para uma rede neural
- O que é **One-hot Encoding** e por que ele é necessário
- Como a multiplicação de matrizes funciona como uma consulta de pesos
- O que são **logits** e como a função **Softmax** os converte em probabilidades
- Como medir o erro do modelo usando **Entropia Cruzada** (*Cross-Entropy Loss*)
- Como os pesos são ajustados automaticamente usando **gradiente descendente**


---
### 1. Carregando as dependências e o Dataset

Importaremos o `torch` (PyTorch) e a biblioteca gráfica `matplotlib` para visualizações.

Em seguida, carregaremos o mesmo **corpus** do notebook anterior.

In [7]:
# Importando bibliotecas
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Garantir reprodutibilidade
torch.manual_seed(42)

# Carregando o mesmo dataset do notebook 01
with open("../../data/input.txt", "r", encoding="utf-8") as f:
    text = f.read().strip()

print("Corpus original:")
print(repr(text))

Corpus original:
'ola mundo\no gato dormiu\no cachorro latiu\nola chatgpt'


#### Exercício de Reflexão

Observe o corpus carregado.

Perguntas:

- Quantas linhas de texto existem no corpus?
- Quantos caracteres únicos você consegue identificar a olho nu?
- Como esse corpus se compara em tamanho a um dataset real (ex: Wikipédia)?


#### Conceito Importante

**Corpus** (do latim, *corpo*) é o nome dado ao conjunto de textos utilizado para treinar ou avaliar um modelo de linguagem.

Exemplos de corpora reais:

- **GPT-4**: treinado com textos na ordem de trilhões de tokens
- **BERT**: Wikipédia em inglês + livros digitalizados (~3 bilhões de palavras)

Nosso corpus tem algumas dezenas de tokens — o objetivo é entender o mecanismo, não escalar o desempenho.

---
### 2. Separando Treino e Teste

Reutilizamos o mesmo split do notebook anterior: **75% treino, 25% teste**.

Isso garante que a comparação entre o modelo clássico e o neural seja feita nas mesmas condições.

In [8]:
lines = text.split('\n')
split_idx = int(len(lines) * 0.75)

train_lines = lines[:split_idx]
test_lines  = lines[split_idx:]

train_text = '\n'.join(train_lines)
test_text  = '\n'.join(test_lines)

print(f"Linhas de Treino:\n{train_lines}")
print(f"Linhas de Teste:\n{test_lines}")

Linhas de Treino:
['ola mundo', 'o gato dormiu', 'o cachorro latiu']
Linhas de Teste:
['ola chatgpt']


#### Exercício de Reflexão

Observe os conjuntos gerados.

Perguntas:

- Por que usamos o **mesmo split** do notebook anterior em vez de definir um novo?
- O que aconteceria na comparação final se os conjuntos de treino fossem diferentes?


#### Conceito Importante

Para comparar dois modelos de forma justa, ambos devem ser treinados e avaliados nos **mesmos dados**.

Qualquer diferença no split introduz uma variável confundidora: não saberíamos se a diferença de desempenho vem do modelo ou dos dados.

---
### 3. Vocabulário e Tokenização

Reutilizamos o mesmo vocabulário e as mesmas funções `encode`/`decode` do notebook anterior, incluindo o token especial de início `^`.

In [9]:
START = '^'
chars = [START] + sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}  # caractere -> ID
itos = {i: ch for i, ch in enumerate(chars)}  # ID -> caractere

encode = lambda s:   [stoi[c] for c in s]
decode = lambda ids: ''.join([itos[i] for i in ids])

print(f"Vocabulário ({vocab_size} elementos):\n{chars}")

Vocabulário (17 elementos):
['^', '\n', ' ', 'a', 'c', 'd', 'g', 'h', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 't', 'u']


#### Exercício de Reflexão

Observe o vocabulário e seu tamanho.

Perguntas:

- O vocabulário é idêntico ao do notebook anterior? Por quê?
- O que aconteceria se um token do conjunto de teste não estivesse presente no vocabulário?


#### Conceito Importante

Tokens presentes no teste mas ausentes do vocabulário de treino são chamados de **OOV** (*Out-Of-Vocabulary*).

Para evitar esse problema, construímos o vocabulário a partir do **texto completo** (treino + teste juntos).

Modelos modernos usam técnicas como **BPE** (*Byte-Pair Encoding*) e **WordPiece** que minimizam o OOV ao dividir palavras desconhecidas em subunidades menores já conhecidas.

---
### 4. Construindo o Dataset de Pares para a Rede Neural

Para treinar uma rede neural, precisamos organizar os dados em **pares (entrada, alvo)**.

No caso do bigrama, a tarefa é sempre a mesma: dado o token atual, prever o próximo.
Para cada sequência (ex: `^ola\n`) geramos os pares:

- Dado `^` → prever `o`
- Dado `o` → prever `l`
- Dado `l` → prever `a`
- Dado `a` → prever `\n`

Vamos armazenar esses pares em dois **tensores** do PyTorch: `xs` (entradas) e `ys` (alvos).

---

#### O que é um Tensor?

Um **tensor** é a forma como o PyTorch representa arrays de números.

- Uma lista comum de Python: `[3, 12, 9]`
- O equivalente como tensor: `torch.tensor([3, 12, 9])`

A vantagem do tensor é que o PyTorch sabe realizar operações matemáticas sobre ele de forma eficiente (incluindo na GPU) e, mais importante para nós, consegue **calcular gradientes automaticamente** — algo que exploraremos nas seções seguintes.

In [10]:
xs = []
ys = []

for line in train_lines:
    # Encodamos a linha adicionando START no início e o final de linha '\n' no fim
    encoded = encode(START + line + '\n')
    for i in range(len(encoded) - 1):
        xs.append(encoded[i])
        ys.append(encoded[i+1])

# Convertendo as listas de IDs em tensores do PyTorch (tipo inteiro de 64 bits)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num_examples = xs.nelement()

print(f"Número total de exemplos de treino: {num_examples}")
print("Primeiros 5 exemplos:")
for i in range(5):
    print(f"Exemplo {i}: entrada {repr(itos[xs[i].item()])} (ID {xs[i]}) -> alvo {repr(itos[ys[i].item()])} (ID {ys[i]})")

Número total de exemplos de treino: 41
Primeiros 5 exemplos:
Exemplo 0: entrada '^' (ID 0) -> alvo 'o' (ID 12)
Exemplo 1: entrada 'o' (ID 12) -> alvo 'l' (ID 9)
Exemplo 2: entrada 'l' (ID 9) -> alvo 'a' (ID 3)
Exemplo 3: entrada 'a' (ID 3) -> alvo ' ' (ID 2)
Exemplo 4: entrada ' ' (ID 2) -> alvo 'm' (ID 10)


#### Exercício de Reflexão

Observe os primeiros 5 pares de treino.

Perguntas:

- O par `'^' → 'o'` faz sentido? Por que é o primeiro?
- Quantos pares você esperaria para as 3 linhas de treino? Confere com `num_examples`?
- Se o corpus tivesse 1 milhão de linhas, o tensor `xs` caberia na memória RAM de um computador comum?


#### Conceito Importante

Os tensores `xs` (entradas) e `ys` (alvos) formam o **dataset supervisionado** da rede neural.

| Tensor | Significado |
|--------|-------------|
| `xs[i]` | ID do token atual (o que o modelo vê) |
| `ys[i]` | ID do próximo token (o que o modelo deve prever) |

Toda rede neural para linguagem aprende a partir de pares `(entrada → alvo)`. A complexidade dos modelos modernos está em *como* eles processam essa entrada — não na estrutura básica do problema.

---
### 5. One-Hot Encoding

Os IDs em `xs` são números inteiros (como `0, 12, 9, 3`). Se passarmos esses IDs diretamente para uma rede neural para serem multiplicados por pesos, a rede assumirá que o ID `12` é doze vezes maior que o ID `1`, o que não faz sentido já que a ordem dos caracteres no vocabulário é arbitrária.

Para resolver isso, representamos cada ID como um vetor binário de tamanho `vocab_size` com apenas uma posição igual a `1` (a correspondente ao ID) e todas as outras iguais a `0`. Esse processo é chamado de **One-hot Encoding**.

Por exemplo, com um vocabulário de tamanho 5, o ID `3` vira o vetor `[0, 0, 0, 1, 0]`.

In [11]:
# Codificando xs em one-hot encodings
# F.one_hot gera tensores inteiros, por isso convertemos para float32 para operações de rede neural
xenc = F.one_hot(xs, num_classes=vocab_size).float()

print(f"Shape de xenc (Exemplos, Vocabulário): {xenc.shape}")
print(f"Representação one-hot do primeiro exemplo (entrada {repr(itos[xs[0].item()])}):\n{xenc[0]}")

Shape de xenc (Exemplos, Vocabulário): torch.Size([41, 17])
Representação one-hot do primeiro exemplo (entrada '^'):
tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


#### Exercício de Reflexão

Observe o vetor one-hot do primeiro exemplo.

Perguntas:

- Quantos `1`s e quantos `0`s existem no vetor? O que isso diz sobre a esparsidade?
- Por que não podemos passar o ID inteiro diretamente para a rede?
- Com um vocabulário de 50.000 tokens, quanto de memória cada vetor one-hot ocuparia?


#### Conceito Importante

Passar IDs inteiros diretamente para a rede criaria um problema de **assunção ordinal**: a rede interpretaria que o ID `12` tem valor *doze vezes maior* que o ID `1`.

Mas a ordem dos caracteres no vocabulário é **arbitrária** — não há hierarquia entre `'a'` e `'b'`.

O one-hot encoding elimina essa suposição incorreta ao tratar cada token como uma categoria independente.

> Nos próximos notebooks estudaremos **embeddings** — representações densas que substituem o one-hot de forma muito mais eficiente e que permitem capturar *similaridades* entre tokens.

---
### 6. Estrutura da Rede Neural: Pesos e Forward Pass

Nossa rede neural tem uma única camada: uma matriz de pesos `W` com dimensões `(vocab_size, vocab_size)`.

Agora vamos ver o que acontece, passo a passo, desde o vetor one-hot até as probabilidades finais.

---

#### Passo 1 — De one-hot para Logits (via multiplicação por W)

Recebemos do passo anterior um vetor one-hot — por exemplo, para o token de ID 2 num vocabulário de tamanho 3:

```
one-hot = [0, 0, 1]   ← só o índice 2 está ligado
```

A matriz de pesos `W` contém números reais aprendidos durante o treino. Cada **linha** de `W` guarda as pontuações brutas para um token de entrada:

```
         ┌ próximo token 0   próximo token 1   próximo token 2 ┐
W  =  linha 0:  [ 0.5,             -1.2,              0.8 ]
      linha 1:  [ 2.1,              0.3,             -0.5 ]
      linha 2:  [ 1.0,              2.0,              0.5 ]  ← esta linha corresponde ao token 2
```

Quando multiplicamos `[0, 0, 1] @ W`, o que acontece?

```
[0, 0, 1] @ W  =  0 × linha0  +  0 × linha1  +  1 × linha2
               =  [0, 0, 0]   +  [0, 0, 0]   +  [1.0, 2.0, 0.5]
               =  [1.0, 2.0, 0.5]
```

O vetor one-hot age como um **seletor**: o único `1` (na posição 2) extrai exatamente a linha 2 de `W`.

O resultado — `[1.0, 2.0, 0.5]` — são os **logits**: as pontuações brutas que a rede atribui a cada possível próximo token.

**O que são logits?**

Logits são os valores "brutos" da rede antes de qualquer normalização. Um logit alto significa que a rede acha aquele próximo token mais provável; um logit baixo significa o contrário. Mas ainda não são probabilidades — podem ser negativos, maiores que 1, qualquer valor real.

---

#### Passo 2 — De Logits para Probabilidades (Softmax)

Precisamos converter os logits `[1.0, 2.0, 0.5]` em probabilidades válidas (valores entre 0 e 1 que somam 1).

Para isso usamos a **Softmax**. O nome vem de *soft maximum*: em vez de simplesmente apontar o maior valor (máximo "duro"), ela distribui a probabilidade de forma suave, dando mais peso aos valores maiores sem zerar os menores.

A Softmax opera em dois sub-passos:

**2a. Exponenciação** — calculamos $e^{\text{logit}}$ para cada valor:

O número $e \approx 2{,}718$ é a base dos logaritmos naturais. Ele tem uma propriedade chave: $e^x$ é **sempre positivo**, mesmo quando $x$ é negativo — o que resolve o problema de logits negativos.

**2b. Normalização** — dividimos cada resultado pela soma de todos:

$$P_i = \frac{e^{\text{logit}_i}}{\sum_j e^{\text{logit}_j}}$$

Isso garante que a soma de todas as probabilidades seja exatamente **1.0**.

---

#### Exemplo completo — do one-hot às probabilidades

```
one-hot  =  [0,    0,    1   ]   ← token 2 está ativo
         × W
           ─────────────────────
logits   =  [1.0,  2.0,  0.5 ]   ← linha 2 de W (pontuações brutas)

exp      =  [e^1.0, e^2.0, e^0.5]
         =  [2.72,  7.39,  1.65 ]   → soma = 11.76

probs    =  [2.72/11.76, 7.39/11.76, 1.65/11.76]
         =  [0.23,       0.63,       0.14       ]   → soma = 1.0
```

O token 1 tinha o maior logit (2.0) → recebe a maior probabilidade (0.63) após a Softmax.

In [12]:
# Inicializando os pesos W com valores aleatórios seguindo uma distribuição normal.
# require_grad=True sinaliza ao PyTorch que precisamos calcular gradientes para essa matriz.
g = torch.Generator().manual_seed(42)
W = torch.randn((vocab_size, vocab_size), generator=g, requires_grad=True)

# Exemplo de Forward Pass simplificado para todos os xs:
logits = xenc @ W # Multiplicação matricial
counts = logits.exp() # Exponencial das pontuações (simulando contagens positivas)
probs = counts / counts.sum(1, keepdims=True) # Softmax feito manualmente

print(f"Shape de probs: {probs.shape}")
print(f"Exemplo de probabilidade de saída para o 1º caractere (soma = {probs[0].sum().item():.2f}):\n{probs[0]}")

Shape de probs: torch.Size([41, 17])
Exemplo de probabilidade de saída para o 1º caractere (soma = 1.00):
tensor([0.2099, 0.1352, 0.0752, 0.0037, 0.0602, 0.0089, 0.0293, 0.0061, 0.0144,
        0.1589, 0.0206, 0.0075, 0.0148, 0.0175, 0.0142, 0.0655, 0.1579],
       grad_fn=<SelectBackward0>)


#### Exercício de Reflexão

Observe o shape de `probs` e a soma das probabilidades do primeiro exemplo.

Perguntas:

- Por que `probs` tem shape `(N, vocab_size)` e não apenas `(vocab_size,)`?
- O que cada linha de `W` representa intuitivamente? Compare com a tabela de bigramas do notebook 01.
- O que acontece com as probabilidades se inicializarmos `W` com zeros em vez de valores aleatórios? Experimente.


#### Conceito Importante

A matriz `W` desempenha o mesmo papel que a **tabela de bigramas** do notebook anterior — mas em vez de ser preenchida por contagens, ela é *aprendida por otimização*.

Cada linha de `W` corresponde a um token de entrada e contém as pontuações brutas (*logits*) para cada possível próximo token.

Após o treinamento, veremos que as probabilidades geradas por `W` convergem para as mesmas probabilidades da tabela de contagens clássica — isso é demonstrado na seção 10.

---
### 7. A Função de Perda (Cross-Entropy Loss)

Queremos que o modelo atribua **alta probabilidade** ao token correto. Mas como medir numericamente o quão bem (ou mal) ele está fazendo isso?

---

#### O Logaritmo como Medida de Surpresa

Usamos o **logaritmo natural** ($\log$ ou $\ln$) da probabilidade para isso.

A intuição é simples:

- $\log(1.0) = 0$ → o modelo previu com certeza absoluta → **sem surpresa**
- $\log(0.5) \approx -0.69$ → o modelo estava 50% certo → **surpresa moderada**
- $\log(0.01) \approx -4.6$ → o modelo atribuiu probabilidade baixíssima → **grande surpresa**

Quanto menor a probabilidade atribuída ao token correto, mais negativo o logaritmo — ou seja, maior a surpresa.

---

#### A Fórmula da Cross-Entropy Loss

Invertemos o sinal (para transformar algo negativo em positivo) e tiramos a média sobre todos os exemplos:

$$\text{Loss} = -\frac{1}{N} \sum_{i=1}^{N} \log\bigl(P(\text{alvo}_i \mid \text{entrada}_i)\bigr)$$

O nome **Entropia Cruzada** vem da teoria da informação — é uma medida de quão diferente a distribuição prevista pelo modelo é da distribuição real (onde a probabilidade do token correto deveria ser 1.0).

O objetivo do treinamento é **minimizar essa loss** — fazer o modelo ficar cada vez menos surpreso com os dados reais.

In [13]:
# Para cada exemplo i, queremos extrair probs[i, ys[i]]
# O PyTorch permite fazer isso indexando de forma simples:
correct_probs = probs[torch.arange(num_examples), ys]

# Calculamos a perda como a média do logaritmo negativo das probabilidades corretas
loss = -correct_probs.log().mean()
print(f"Perda calculada manualmente: {loss.item():.4f}")

# O PyTorch possui uma função nativa mais otimizada numericamente (evita problemas de underflow/overflow):
# F.cross_entropy recebe os logits brutas e os alvos ys diretamente
loss_native = F.cross_entropy(logits, ys)
print(f"Perda calculada pela função nativa do PyTorch: {loss_native.item():.4f}")

Perda calculada manualmente: 3.2347
Perda calculada pela função nativa do PyTorch: 3.2347


#### Exercício de Reflexão

Observe os dois valores de loss (manual e nativo do PyTorch).

Perguntas:

- Os valores são iguais. O que isso confirma sobre a implementação manual?
- Se a rede atribuísse probabilidade `1.0` ao token correto em todos os exemplos, qual seria a loss?
- Se atribuísse probabilidade `0.0`, o que aconteceria matematicamente?


#### Conceito Importante

A **Cross-Entropy Loss** penaliza o modelo proporcionalmente ao quão errada é sua previsão:

| Prob. do token correto | Loss |
|------------------------|------|
| 1.0 — perfeito | 0.0 |
| 0.5 — incerto | ≈ 0.69 |
| → 0.0 — completamente errado | → ∞ |

Minimizar a loss é matematicamente equivalente a **maximizar a probabilidade atribuída aos tokens corretos** — princípio chamado de *Maximum Likelihood Estimation* (MLE).

---
### 8. O Loop de Treinamento (Gradiente Descendente)

Agora temos tudo o que precisamos. A cada iteração do treinamento fazemos 4 passos:

---

#### O que é Gradiente?

O **gradiente** de uma função em relação a um parâmetro indica a direção e a intensidade com que a função aumenta quando mudamos esse parâmetro.

Pense em uma esfera rolando numa superfície irregular. O gradiente aponta morro acima — então para descer (minimizar a loss), andamos na **direção oposta** ao gradiente.

O PyTorch calcula o gradiente de todos os pesos de `W` automaticamente com `loss.backward()`. Você não precisa calcular nenhuma derivada manualmente.

---

#### Os 4 Passos do Loop

1. **Forward Pass**: calculamos os logits e a loss com os pesos atuais
2. **Resetar Gradientes**: zeramos os gradientes da iteração anterior (`W.grad = None`)
3. **Backward Pass**: o PyTorch percorre o grafo de operações ao contrário e calcula quanto cada peso contribuiu para a loss (`loss.backward()`)
4. **Atualizar Pesos**: movemos cada peso na direção oposta ao seu gradiente, escalado pela **taxa de aprendizado** (*learning rate*):

$$W \leftarrow W - \text{lr} \times \nabla W$$

A **taxa de aprendizado** controla o tamanho do passo: muito grande e o modelo oscila sem convergir; muito pequena e o treinamento fica lento.

In [14]:
# Redefinindo o peso para treinar do zero
W = torch.randn((vocab_size, vocab_size), generator=g, requires_grad=True)

epochs = 200
learning_rate = 50.0  # Usamos uma learning rate alta pois o dataset é muito pequeno e a rede é muito simples

print("Iniciando o loop de treinamento...")
print("--------------------------------")

for epoch in range(epochs):
    # 1. Forward Pass
    logits = xenc @ W
    loss = F.cross_entropy(logits, ys)
    
    # 2. Resetar os gradientes
    W.grad = None
    
    # 3. Backward Pass
    loss.backward()
    
    # 4. Atualizar os pesos (usamos W.data para evitar que o PyTorch rastreie essa operação no grafo de gradientes)
    W.data += -learning_rate * W.grad
    
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"Época {epoch + 1:3d}/{epochs} | Loss: {loss.item():.4f}")

Iniciando o loop de treinamento...
--------------------------------
Época   1/200 | Loss: 3.1091
Época  20/200 | Loss: 1.1057
Época  40/200 | Loss: 1.0988
Época  60/200 | Loss: 1.1233
Época  80/200 | Loss: 1.1260
Época 100/200 | Loss: 1.1257
Época 120/200 | Loss: 1.1255
Época 140/200 | Loss: 1.1253
Época 160/200 | Loss: 1.1251
Época 180/200 | Loss: 1.1250
Época 200/200 | Loss: 1.1250


#### Exercício de Reflexão

Observe a evolução da loss ao longo das épocas.

Experimente:

- Altere `learning_rate` para `0.1` e depois para `500.0`. O que acontece com a convergência?
- Aumente `epochs` para `1000`. A loss continua caindo ou estabiliza?
- Por que zeramos `W.grad = None` no início de cada iteração em vez de deixar acumular?


#### Conceito Importante

O loop de treinamento segue sempre o mesmo ciclo de 4 passos:

| Passo | O que acontece |
|-------|----------------|
| **Forward Pass** | A rede calcula as probabilidades com os pesos atuais |
| **Loss** | Medimos o erro entre as previsões e os alvos reais |
| **Backward Pass** | O PyTorch calcula os gradientes automaticamente |
| **Update** | Ajustamos os pesos na direção que reduz a loss |

Esse ciclo de 4 passos é a base de **todo** treinamento de redes neurais — de uma camada linear até os maiores Transformers.

---
### 9. Geração de Texto Autoregressiva

Para gerar texto a partir do modelo treinado:
1. Iniciamos a sequência com o ID do token de início `^` (`START`).
2. Alimentamos esse token na rede neural, obtendo as probabilidades para o próximo caractere.
3. Amostramos o próximo caractere a partir dessas probabilidades usando `torch.multinomial`.
4. Anexamos o caractere amostrado à sequência e repetimos o processo usando o novo caractere como entrada.
5. O processo para quando geramos o token especial de fim de linha `\n`.

In [15]:
print("Texto gerado pelo modelo neural:")
print("-------------------------------")

g_gen = torch.Generator().manual_seed(42)

for _ in range(5):
    out = []
    # Começa com o token de início '^'
    ix = stoi[START]
    
    while True:
        # Passa o token atual pela rede
        # 1. One-hot do ID atual
        x_input = F.one_hot(torch.tensor([ix]), num_classes=vocab_size).float()
        # 2. Forward pass
        logits_curr = x_input @ W
        # 3. Softmax
        p_curr = F.softmax(logits_curr, dim=1)
        
        # Amostrar a partir das probabilidades preditas
        ix = torch.multinomial(p_curr, num_samples=1, replacement=True, generator=g_gen).item()
        
        # Se for o fim de linha '\n', paramos a geração
        if ix == stoi['\n']:
            break
            
        out.append(itos[ix])
        
    print(''.join(out))

Texto gerado pelo modelo neural:
-------------------------------
olato
olatolatola ca miu
ola dola latiu
olato cholaca dolacholacholatiu
ola cacatolachola mundolatiu


#### Exercício de Reflexão

Observe o texto gerado.

Perguntas:

- O resultado se parece com o texto gerado no notebook anterior? Por quê?
- Por que usamos `torch.multinomial` (amostragem probabilística) em vez de simplesmente escolher o token com maior probabilidade?
- O que aconteceria se gerarmos após apenas 10 épocas de treino?


#### Conceito Importante

A geração autoregressiva é **idêntica em espírito** à do notebook anterior: um token por vez, usando a saída atual como entrada do próximo passo.

A diferença está na **origem das probabilidades**:

| Notebook 01 | Notebook 02 |
|-------------|-------------|
| Consulta tabela de contagens | Passa o token pela rede neural |
| Probabilidades fixas, calculadas uma vez | Probabilidades *aprendidas* iterativamente |

O laço de geração é exatamente o mesmo — o que muda é como as probabilidades foram obtidas.

---
### 10. Comparação Matemática: Rede Neural vs. Tabela de Contagens

Uma rede neural de camada linear linear mapeada por one-hot e Softmax é matematicamente equivalente ao modelo de Bigramas clássico. A única diferença é a forma como chegamos aos pesos:
- No modelo clássico: contamos a frequência e dividimos de forma direta.
- No modelo neural: otimizamos pesos aleatórios até convergir para as proporções ideais.

Vamos comparar as probabilidades obtidas pelo modelo neural final com o modelo clássico com suavização de Laplace para provar essa equivalência.

In [16]:
# --------------------------------------------------
# 1. Calculando as probabilidades na rede neural
# --------------------------------------------------
with torch.no_grad(): # Desabilita rastreamento de gradiente para economizar memória
    # Logits para todos os IDs de 0 a vocab_size-1
    all_inputs = F.one_hot(torch.arange(vocab_size), num_classes=vocab_size).float()
    neural_probs = F.softmax(all_inputs @ W, dim=1)

# --------------------------------------------------
# 2. Calculando as probabilidades clássicas (Suavização Laplace alpha=1)
# --------------------------------------------------
train_data = []
for line in train_lines:
    train_data += encode(START + line + '\n')

bigrams = {}
for i in range(len(train_data) - 1):
    curr, nxt = train_data[i], train_data[i+1]
    if curr not in bigrams: bigrams[curr] = {}
    bigrams[curr][nxt] = bigrams[curr].get(nxt, 0) + 1

classic_probs = torch.zeros((vocab_size, vocab_size))
alpha = 1
for current_token in range(vocab_size):
    counts = bigrams.get(current_token, {})
    total = sum(counts.values()) + alpha * vocab_size
    for next_token in range(vocab_size):
        count = counts.get(next_token, 0)
        classic_probs[current_token, next_token] = (count + alpha) / total

# --------------------------------------------------
# 3. Comparando um caractere específico, ex: depois do START '^'
# --------------------------------------------------
start_idx = stoi[START]
print(f"Distribuição de probabilidades depois do token inicial '{START}':")
print("-----------------------------------------------------------")
print(f"{'Caractere':<12} | {'Modelo Clássico (%)':<20} | {'Modelo Neural (%)':<18}")
for i in range(vocab_size):
    char_repr = repr(itos[i])
    p_classic = classic_probs[start_idx, i].item() * 100
    p_neural = neural_probs[start_idx, i].item() * 100
    print(f"{char_repr:<12} | {p_classic:<20.2f}% | {p_neural:<18.2f}%")

Distribuição de probabilidades depois do token inicial '^':
-----------------------------------------------------------
Caractere    | Modelo Clássico (%)  | Modelo Neural (%) 
'^'          | 5.00                % | 0.01              %
'\n'         | 5.00                % | 0.01              %
' '          | 5.00                % | 0.01              %
'a'          | 5.00                % | 0.01              %
'c'          | 5.00                % | 0.01              %
'd'          | 5.00                % | 0.00              %
'g'          | 5.00                % | 0.00              %
'h'          | 5.00                % | 0.01              %
'i'          | 5.00                % | 0.00              %
'l'          | 5.00                % | 0.01              %
'm'          | 5.00                % | 0.00              %
'n'          | 5.00                % | 0.01              %
'o'          | 20.00               % | 99.87             %
'p'          | 5.00                % | 0.01             

#### Exercício de Reflexão

Compare as colunas do Modelo Clássico e do Modelo Neural.

Perguntas:

- As probabilidades são idênticas ou apenas *próximas*? Por quê a diferença?
- O que aconteceria se treinássemos por 1000 épocas? As probabilidades convergiriam ainda mais?
- Qual técnica do modelo clássico (suavização de Laplace) tem efeito equivalente à regularização L2 nos pesos da rede?


#### Discussão da Equivalência

Observe que as probabilidades do **Modelo Neural** e do **Modelo Clássico** são extremamente parecidas!

Se treinássemos a rede neural por mais épocas ou com parâmetros específicos de suavização (como regularização L2 de pesos), elas seriam ainda mais próximas. 

Isso prova que a rede neural aprendeu a mesma distribuição estatística a partir dos dados, mas em vez de contar deterministicamente, ela encontrou a resposta através de otimização numérica de pesos.

---
### Resumo

Neste notebook aprendemos:

✅ Como estruturar um dataset como pares de tokens para treinamento

✅ O que é o One-hot encoding e sua importância na conversão de IDs categóricos

✅ Como o forward pass de uma única camada linear consiste em multiplication matricial (`xenc @ W`) e Softmax

✅ Como calcular a perda de Entropia Cruzada (*Cross-Entropy Loss*) e sua intuição matemática

✅ Como realizar o ciclo completo de treinamento via retropropagação de gradientes no PyTorch

✅ Como amostrar de forma probabilística para gerar texto autorregressivamente

---
### Desafios Opcionais

Experimente:
- Alterar a Taxa de Aprendizado (`learning_rate`) de `50.0` para `0.1` ou `500.0`. O que acontece com a Loss e a velocidade de convergência?
- Alterar o número de épocas para `1000` e observar se o modelo neural converge para valores ainda mais próximos do modelo de contagem clássico.
- Tentar adicionar uma penalidade L2 aos pesos (também chamada de *weight decay* na otimização) para ver o efeito equivalente de suavização nas probabilidades preditas.

Próximo passo: **03 — Markov** (expandindo o contexto de tamanho 1 para ordem N usando cadeias de Markov).